# Predictive Power Score (PPS) & SHAP Feature Importance

## Purpose
Identify asymmetric predictive relationships between survey variables using Predictive Power Score (PPS), a metric that goes beyond traditional correlation by detecting non-linear and asymmetric dependencies. Complement PPS with SHAP (SHapley Additive exPlanations) values to understand feature importance in predicting key outcomes.

## Why PPS Over Correlation?
- **Correlation** is symmetric: cor(X, Y) = cor(Y, X)
- **PPS** is asymmetric: X may predict Y well, but Y may not predict X
- **Correlation** only detects linear relationships
- **PPS** detects any predictive pattern (linear, non-linear, categorical)
- PPS ranges from 0 (no predictive power) to 1 (perfect prediction)

## PPS Formula
- Classification: PPS = (F1_model - F1_naive) / (1 - F1_naive)
- Regression: PPS = 1 - (MAE_model / MAE_naive)

## Steps
1. Load survey response data
2. Select and recode survey items to numeric
3. Compute PPS matrix across all variable pairs
4. Visualize PPS heatmap
5. Fit a Random Forest on a key outcome variable
6. Compute and visualize SHAP feature importance
7. (Optional) Subgroup analysis: compare PPS across organizational units

## Interpretation Guide
- **PPS = 0**: Variable X has no predictive power for Y
- **PPS = 1**: X perfectly predicts Y
- **PPS > 0.2**: Meaningful predictive relationship worth investigating
- **Asymmetric PPS**: If PPS(X→Y) >> PPS(Y→X), X is a leading indicator of Y
- **SHAP values**: Show each feature's contribution to predictions; positive SHAP pushes prediction up, negative pushes down

In [ ]:
# ---- CONFIGURATION ----
DATA_PATH = "synthetic_survey_responses.csv"

# Target variable for SHAP analysis (the outcome you want to explain)
TARGET_VARIABLE = "Q1_1"  # e.g., "Would recommend as great place to work"

# Subgroup column for optional comparison analysis
SUBGROUP_COLUMN = "department"  # Set to None to skip subgroup analysis

In [ ]:
# Step 1: Import Libraries
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ppscore as pps

pd.set_option('display.max_colwidth', 200)
%matplotlib inline

print("Libraries loaded.")

In [ ]:
# Step 2: Load and Prepare Data
df = pd.read_csv(DATA_PATH)
print(f"Dataset: {df.shape[0]} rows x {df.shape[1]} columns")

# Select numeric survey items for PPS analysis
survey_cols = [c for c in df.columns if c.startswith('Q') or c.startswith('WB_')]
print(f"\nSurvey items selected: {len(survey_cols)}")
print(survey_cols)

df_subset = df[survey_cols].copy()
df_subset.head()

In [ ]:
# Step 3: Compute PPS Matrix
# This computes the predictive power score for every pair of variables.
# For N variables, this creates an N x N matrix where entry (i,j) shows
# how well variable i predicts variable j.

print("Computing PPS matrix (this may take a minute)...")
pps_matrix = pps.matrix(df_subset)
print(f"\nPPS results: {pps_matrix.shape[0]} pairs computed")
pps_matrix.head(10)

In [ ]:
# Step 4: PPS Heatmap Visualization
# Pivot the results into a square matrix for heatmap display

pps_pivot = pps_matrix[['x', 'y', 'ppscore']].pivot(
    columns='x', index='y', values='ppscore'
)

plt.figure(figsize=(16, 12))
sns.set_style("whitegrid")
sns.heatmap(
    pps_pivot,
    vmin=0, vmax=1,
    cmap="Blues",
    linewidths=0.5,
    annot=True,
    fmt='.2f',
    annot_kws={"fontsize": 7},
    xticklabels=True,
    yticklabels=True
)
plt.title('Predictive Power Score Matrix\n(Row predicts Column)', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Step 5: Top Predictive Pairs
# Identify the strongest asymmetric relationships

# Filter out self-predictions (PPS of a variable with itself = 1)
top_pairs = pps_matrix[pps_matrix['x'] != pps_matrix['y']].copy()
top_pairs = top_pairs.sort_values('ppscore', ascending=False)

print("Top 20 Predictive Pairs (strongest PPS):")
print(top_pairs[['x', 'y', 'ppscore']].head(20).to_string(index=False))

# Check for asymmetric relationships
print("\n\nAsymmetric Relationships (large PPS difference between directions):")
pairs_wide = top_pairs.pivot(index='x', columns='y', values='ppscore')
for i, row in top_pairs.head(20).iterrows():
    x, y, score = row['x'], row['y'], row['ppscore']
    reverse = top_pairs[(top_pairs['x'] == y) & (top_pairs['y'] == x)]['ppscore']
    if len(reverse) > 0:
        rev_score = reverse.values[0]
        diff = abs(score - rev_score)
        if diff > 0.05:
            print(f"  {x} → {y}: {score:.3f}  vs  {y} → {x}: {rev_score:.3f}  (diff: {diff:.3f})")

In [ ]:
# Step 6: Pairwise PPS for a Specific Target
# Which variables best predict our target outcome?

print(f"\nPPS scores predicting '{TARGET_VARIABLE}':")
target_pps = pps_matrix[pps_matrix['y'] == TARGET_VARIABLE].sort_values('ppscore', ascending=False)
target_pps_clean = target_pps[target_pps['x'] != TARGET_VARIABLE]
print(target_pps_clean[['x', 'ppscore']].head(15).to_string(index=False))

# Bar chart
plt.figure(figsize=(10, 6))
top_predictors = target_pps_clean.head(15)
plt.barh(range(len(top_predictors)), top_predictors['ppscore'].values, color='steelblue')
plt.yticks(range(len(top_predictors)), top_predictors['x'].values)
plt.xlabel('Predictive Power Score')
plt.title(f'Top Predictors of {TARGET_VARIABLE}')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Step 7: SHAP Feature Importance with Random Forest
# SHAP values show each feature's marginal contribution to predictions.
# This complements PPS by showing directionality (positive vs negative impact).

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# Prepare data for classification
# Binarize target: above median = 1, at or below = 0
median_val = df_subset[TARGET_VARIABLE].median()
y_binary = (df_subset[TARGET_VARIABLE] > median_val).astype(int)
X_shap = df_subset.drop(columns=[TARGET_VARIABLE]).dropna()
y_binary = y_binary[X_shap.index]

X_train, X_test, y_train, y_test = train_test_split(
    X_shap, y_binary, test_size=0.25, random_state=42
)

# Fit Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

accuracy = rf_model.score(X_test, y_test)
print(f"Random Forest accuracy: {accuracy:.4f}")

In [ ]:
# Step 8: SHAP Values
try:
    import shap
    
    explainer = shap.TreeExplainer(rf_model)
    shap_values = explainer.shap_values(X_train)
    
    # Summary plot: shows feature importance + direction of effect
    print("SHAP Summary Plot:")
    print("  Each dot = one observation")
    print("  X-axis = SHAP value (impact on model output)")
    print("  Color = feature value (red = high, blue = low)")
    shap.summary_plot(shap_values[1], X_train)  # Class 1 = above median
    
except ImportError:
    print("SHAP not installed. Install with: pip install shap")
    print("Falling back to sklearn feature importances...")
    
    importances = pd.DataFrame({
        'feature': X_shap.columns,
        'importance': rf_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    plt.figure(figsize=(10, 6))
    plt.barh(range(15), importances['importance'].head(15), color='steelblue')
    plt.yticks(range(15), importances['feature'].head(15))
    plt.xlabel('Feature Importance (Gini)')
    plt.title(f'Random Forest Feature Importance for {TARGET_VARIABLE}')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

In [ ]:
# Step 9: Variance Inflation Factor (VIF)
# Checks for multicollinearity among predictors.
# VIF > 5 suggests concerning multicollinearity.
# VIF > 10 indicates severe multicollinearity.

from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_data = df_subset.dropna()
vif_results = pd.DataFrame()
vif_results["feature"] = vif_data.columns
vif_results["VIF"] = [
    variance_inflation_factor(vif_data.values, i)
    for i in range(vif_data.shape[1])
]

vif_results = vif_results.sort_values('VIF', ascending=False)
print("Variance Inflation Factors:")
print("  VIF > 5: concerning multicollinearity")
print("  VIF > 10: severe multicollinearity")
print(vif_results.to_string(index=False))

In [ ]:
# Step 10: Subgroup PPS Comparison (Optional)
# Compare PPS patterns across organizational subgroups to identify
# whether predictive relationships differ by department, level, etc.

if SUBGROUP_COLUMN is not None and SUBGROUP_COLUMN in df.columns:
    subgroups = df[SUBGROUP_COLUMN].unique()[:3]  # Limit to first 3 for speed
    
    def compute_subgroup_pps(subgroup_name):
        """Compute PPS matrix for a specific subgroup."""
        mask = df[SUBGROUP_COLUMN] == subgroup_name
        sub_df = df_subset[mask].copy()
        if len(sub_df) < 30:
            print(f"  Skipping {subgroup_name} (n={len(sub_df)}, too small)")
            return None
        print(f"  Computing PPS for {subgroup_name} (n={len(sub_df)})...")
        sub_pps = pps.matrix(sub_df)
        sub_pivot = sub_pps[['x', 'y', 'ppscore']].pivot(
            columns='x', index='y', values='ppscore'
        )
        return sub_pivot
    
    print(f"\nSubgroup analysis by '{SUBGROUP_COLUMN}':")
    for sg in subgroups:
        result = compute_subgroup_pps(sg)
        if result is not None:
            plt.figure(figsize=(12, 9))
            sns.heatmap(result, vmin=0, vmax=1, cmap="Blues",
                       linewidths=0.5, annot=True, fmt='.2f',
                       annot_kws={"fontsize": 6})
            plt.title(f'PPS Matrix: {sg}', fontsize=14)
            plt.xticks(rotation=45, ha='right')
            plt.tight_layout()
            plt.show()
else:
    print("Subgroup analysis skipped (SUBGROUP_COLUMN not set or not found).")

## Summary

This notebook demonstrated:

1. **PPS Matrix**: Revealed which survey items have the strongest asymmetric predictive relationships
2. **Target-specific PPS**: Identified the top predictors of a specific outcome variable
3. **SHAP Analysis**: Showed not just importance but direction of each feature's impact
4. **VIF**: Flagged multicollinearity issues that could affect regression-based analyses
5. **Subgroup Comparison**: Tested whether predictive patterns vary across organizational units

### Key Differences from Correlation:
| Aspect | Correlation | PPS |
|--------|------------|-----|
| Symmetry | cor(X,Y) = cor(Y,X) | PPS(X→Y) ≠ PPS(Y→X) |
| Relationship type | Linear only | Any pattern |
| Data types | Numeric only | Numeric + categorical |
| Range | -1 to 1 | 0 to 1 |
| Interpretation | Strength + direction | Predictive power only |